# sisl · SIESTA matrices and visualization

This tutorial introduces how to analyze SIESTA calculation results using the Python package sisl. Starting from a simple bulk BN system, we explore the Hamiltonian (H), overlap (S), and density (DM) matrices and their representations in an atomic orbital basis.

We then use these matrices to calculate and visualize electronic properties, including band structures, band folding, and real-space electron densities.

Run the cells in order in a **Colab CPU runtime**. The small SZP basis and sampling settings are for learning the workflow; quantitative calculations require separate convergence tests.


## 1. Setup

Install **SIESTA 5.4.2 (serial)** and **sisl 0.16.4**. SIESTA runs as a separate executable; Python cells use the Colab kernel. Installation takes a few minutes.

In [ ]:
%%bash
set -euo pipefail
mkdir -p /content/mamba
curl -Ls --fail https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content/mamba bin/micromamba
/content/mamba/bin/micromamba create -y -r /content/mamba -p /content/siesta-env \
    -c conda-forge --override-channels 'siesta=5.4.2=nompi_*'
/content/siesta-env/bin/siesta --version

In [ ]:
%pip install -q "sisl==0.16.4" matplotlib netCDF4

import os
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

from pathlib import Path
from urllib.request import urlretrieve
import subprocess
import numpy as np
import matplotlib.pyplot as plt
import sisl as si

print("sisl:", si.__version__)

## 2. Run BN with SIESTA

`RUN.fdf` sets the SZP basis, k mesh, and SCF options; it includes the lattice and four atomic positions from `STRUCT.fdf`. Keep `B.psf` and `N.psf` in the same directory.

The output options used here are `SaveHS` for H/S, `Write.DM` for DM, and `WriteIonXML` for the numerical orbitals needed on real-space grids.

In [ ]:
work = Path("/content/bn-matrices")
work.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/YHKlab-RGLee/tutorial/main/sisl/01/src/inputs"
for name in ["RUN.fdf", "STRUCT.fdf", "B.psf", "N.psf"]:
    urlretrieve(f"{base}/{name}", work / name)
os.chdir(work)

fdf = si.get_sile("RUN.fdf")
structure = fdf.read_geometry()
print(f"Structure: {structure.na} atoms, two BN layers")
print(Path("RUN.fdf").read_text())

In [ ]:
with Path("siesta.out").open("w") as output:
    subprocess.run(["/content/siesta-env/bin/siesta", "RUN.fdf"],
                   stdout=output, stderr=subprocess.STDOUT, check=True)

log = Path("siesta.out").read_text()
if "SCF Convergence" not in log or "SCF_NOT_CONV" in log:
    raise RuntimeError("SCF did not converge; read siesta.out")
required = ["BN.HSX", "BN.DM", "B.ion.xml", "N.ion.xml"]
missing = [name for name in required if not Path(name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing outputs: {missing}; read siesta.out")
print("SCF complete. Ready to read H, S, and DM.")

## 3. Read the matrices

| Object | Meaning | Used for |
|---|---|---|
| H | $H_{ij}=\langle\phi_i|\hat H|\phi_j\rangle$, in eV | Energies and eigenstates |
| S | $S_{ij}=\langle\phi_i|\phi_j\rangle$ | Overlap of the nonorthogonal basis |
| DM | Occupied-state coefficients | Electron density with the numerical orbitals |

`get_sile("RUN.fdf")` connects the inputs and outputs. H and S come from `BN.HSX`; DM comes from `BN.DM`. H already includes the overlap needed for diagonalization.

**Matrix indices label orbitals.** This SZP basis has 9 orbitals per atom (s, p, and polarization d), giving 36 per cell. `a2o` maps atom indices to their orbitals.

In [ ]:
H = fdf.read_hamiltonian(order=["HSX"])
S = fdf.read_overlap(order=["HSX"])
DM = fdf.read_density_matrix(order=["DM"])
Ef = si.get_sile("BN.HSX").read_fermi_level()
geometry = H.geometry

print(f"{geometry.na} atoms, {geometry.no} orbitals; EF = {Ef:.4f} eV")
for ia, atom in enumerate(geometry.atoms):
    print(f"atom {ia} ({atom.symbol}): orbitals {geometry.a2o(ia, all=True)}")

### See the sparsity pattern

Localized orbitals connect only to nearby orbitals. Following the [sisl workshop's `spy` example](https://github.com/zerothi/ts-tbt-sisl-tutorial/blob/master/TB_02/run.ipynb), mark nonzero entries to see these connections.

`tocsr(dim=0)` extracts H as a SciPy sparse matrix. Its 36 rows are orbitals in the reference cell; its columns include periodic neighbors, grouped in blocks of 36. The plot stretches the vertical axis for readability. Dots show connections, and blank regions show zeros.

In [ ]:
h_sparse = H.tocsr(dim=0)
fig, ax = plt.subplots(figsize=(12, 3.5), constrained_layout=True)
ax.spy(h_sparse, markersize=0.7, aspect="auto", color="tab:blue")
ax.xaxis.tick_bottom()
ax.set(xlabel="Orbital j across stored cells", ylabel="Reference-cell orbital i",
       title=f"H sparsity: {h_sparse.shape[0]} × {h_sparse.shape[1]}")
print(f"Nonzero entries: {h_sparse.count_nonzero() / np.prod(h_sparse.shape):.1%}")
plt.show()

### From periodic connections to energies

At each k point, sisl combines the neighboring-cell blocks into square matrices and solves

$$H(\mathbf k)c_n=E_n(\mathbf k)S(\mathbf k)c_n.$$

`H.Hk()` and `H.Sk()` return the 36 × 36 matrices; `H.eigh()` solves this generalized eigenvalue problem. Even at Γ, these matrices include the neighboring cells. All k coordinates below are fractional reciprocal coordinates, and energies are plotted relative to the saved $E_F$.

In [ ]:
k = [0.0, 0.0, 0.0]  # Γ
Hk = H.Hk(k=k, format="array")
Sk = H.Sk(k=k, format="array")
eigenvalues = H.eigh(k=k)
print("H(Γ), S(Γ):", Hk.shape, Sk.shape)
print("Bands 7–10 at Γ, E − EF [eV]:", np.round(eigenvalues[6:10] - Ef, 3))

## 4. Bands and DOS

### Band structure

`BandStructure` samples Γ–M–K–Γ, and `apply.array.eigh()` solves for the bands along this path. The saved H is sufficient; no new SCF run is needed.

In [ ]:
points = [[0, 0, 0], [0.5, 0, 0], [1/3, 1/3, 0], [0, 0, 0]]
bs = si.BandStructure(H, points, 121, names=["Γ", "M", "K", "Γ"])
bands = bs.apply.array.eigh() - Ef
x = bs.lineark()
ticks, labels = bs.lineartick()

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.plot(x, bands, color="tab:blue", lw=0.8)
ax.set(xticks=ticks, xticklabels=labels, xlim=(x[0], x[-1]), ylim=(-15, 15),
       ylabel="E − EF [eV]", title="BN band structure")
ax.axhline(0, color="gray", ls=":")
ax.grid(axis="x")
plt.show()

### Density of states

DOS averages over a Brillouin-zone mesh instead of a line path. `MonkhorstPack` supplies the k points and weights; `state.DOS()` broadens each state's energies before averaging. For this nonmagnetic H, the result is per spin channel.

In [ ]:
bz = si.MonkhorstPack(H, [6, 6, 2])
energy = np.linspace(-15, 15, 301)
dos = bz.apply.average.eigenstate(wrap=lambda state: state.DOS(energy + Ef))

fig, ax = plt.subplots(figsize=(7, 3), constrained_layout=True)
ax.plot(energy, dos)
ax.set(xlabel="E − EF [eV]", ylabel="DOS [states/eV/cell/spin]")
plt.show()

## 5. Supercell bands

`H.tile(2, axis=0)` doubles the first lattice vector and repeats H and S. The cell now has 72 orbitals and 72 bands, while its first reciprocal vector is halved. This is band folding: the same crystal's states are represented in a smaller reciprocal cell.

Plot only the supercell bands along $\mathbf K=(K_1,0,0)$, with $K_1=-1/2\ldots1/2$ in **supercell reciprocal coordinates**. To compare dispersions on the original-cell k path, one would unfold the supercell states onto that path with their spectral weights.

In [ ]:
H2 = H.tile(2, axis=0)
bs2 = si.BandStructure(H2, [[-0.5, 0, 0], [0.5, 0, 0]], 121)
bands2 = bs2.apply.array.eigh() - Ef
print(f"Orbitals per cell: {H.no} → {H2.no}")

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.plot(bs2.k[:, 0], bands2, color="tab:blue", lw=0.8)
ax.set(xlabel="K₁ (supercell reciprocal coordinates)", ylabel="E − EF [eV]",
       xlim=(-0.5, 0.5), ylim=(-15, 15), title="2× BN supercell: folded bands",
       xticks=[-0.5, 0, 0.5], xticklabels=["−½", "Γ", "+½"])
ax.axhline(0, color="gray", ls=":")
ax.grid(axis="x")
plt.show()

## 6. Electron density and a single state

A `Grid` samples real-space positions. The numerical orbitals in `B.ion.xml` and `N.ion.xml`, attached by the FDF reader, turn orbital coefficients into spatial functions:

- `DM.density(rho)` reconstructs the total electron density, including occupations.
- `state.wavefunction(psi)` reconstructs one selected wavefunction; $|\psi|^2$ shows its probability density.

Choose Γ and `band_index = 7` (the eighth band), the highest occupied band there for this 16-electron nonmagnetic cell. Index 8 selects the first unoccupied band. Use fresh grids because the projection methods add to their contents.

We check the density integral **once** against $2\times3+2\times5=16$ valence electrons. A spacing of 0.15 Å is used throughout; grid-convergence studies are outside this introduction.

In [ ]:
rho = si.Grid(0.15, geometry=DM.geometry)
DM.density(rho)
print(f"Electron count: {rho.grid.sum() * rho.dvolume:.3f} (expected: 16)")

band_index = 7
state = H.eigenstate(k=[0, 0, 0]).sub(band_index)
psi = si.Grid(rho.shape, geometry=H.geometry, dtype=complex)
state.wavefunction(psi)
probability = np.abs(psi.grid)**2
print(f"Γ, band {band_index + 1}: E − EF = {state.eig[0] - Ef:.3f} eV")

### One 2D comparison

Show both quantities on the same plane, about **0.5 Å above the first BN layer**, so out-of-plane orbital weight is visible. Labels mark the projected B/N positions. Cartesian x/y axes preserve the oblique cell shape, and each panel has its own color scale.

In [ ]:
z_slice = geometry.xyz[:, 2].min() + 0.5
iz = round(z_slice / rho.cell[2, 2] * rho.shape[2])
f1, f2 = np.meshgrid(np.arange(rho.shape[0]) / rho.shape[0],
                     np.arange(rho.shape[1]) / rho.shape[1], indexing="ij")
x = f1 * rho.cell[0, 0] + f2 * rho.cell[1, 0]
y = f1 * rho.cell[0, 1] + f2 * rho.cell[1, 1]
layer_atoms = np.flatnonzero(np.isclose(geometry.xyz[:, 2], geometry.xyz[:, 2].min()))

fig, axes = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)
for ax, values, title, unit in zip(
    axes, [rho.grid, probability],
    ["Total electron density", f"Γ, band {band_index + 1}: |ψ|²"],
    ["electrons / Å³", "probability / Å³"],
):
    im = ax.pcolormesh(x, y, values[:, :, iz], shading="nearest", cmap="magma", vmin=0)
    for ia in layer_atoms:
        ax.text(*geometry.xyz[ia, :2], geometry.atoms[ia].symbol, ha="center", va="center",
                color="white", bbox=dict(facecolor="black", alpha=0.5, edgecolor="none"))
    ax.set(xlabel="x [Å]", ylabel="y [Å]", title=title, aspect="equal")
    fig.colorbar(im, ax=ax, label=unit, shrink=0.8)
fig.suptitle(f"Slice at z = {iz * rho.cell[2, 2] / rho.shape[2]:.2f} Å")
plt.show()

## Try it

- Select band index 8 and rerun the two grid cells. How does its spatial pattern differ?
- In the sparsity cell, replace `H.tocsr(dim=0)` with `S.tocsr(dim=0)` or `DM.tocsr(dim=0)` and update the title. Which connections remain?

## References

- [sisl workshop: matrix sparsity with `spy`](https://github.com/zerothi/ts-tbt-sisl-tutorial/blob/master/TB_02/run.ipynb)
- [Hamiltonian: Hk, Sk, eigh, tile](https://zerothi.github.io/sisl/api/generated/sisl.physics.Hamiltonian.html)
- [DensityMatrix: density](https://zerothi.github.io/sisl/api/generated/sisl.physics.DensityMatrix.html)
- [sisl SIESTA tutorial: bands, DOS, and wavefunctions](https://zerothi.github.io/sisl/tutorials/tutorial_siesta_2.html)